This tutorial demonstrates how to build a rank prediction model for Go using PyTorch and WebDataset. We will cover data preparation, model definition, training, validation, and inference. The model will predict the rank of Go players based on game records stored in CSV files. 

PyTorch and WebDataset are powerful tools for handling large datasets and building deep learning models. By the end of this tutorial, you will have a solid understanding of how to implement a rank prediction model for Go. It is advised to run this tutorial on Ubuntu with a GPU for optimal performance.

**This tutorial is for reference purposes only. You are welcome to modify and improve the code as needed**. There is no restriction on the model architecture in this competition. Also, you can use any other libraries or frameworks you prefer.

In [1]:
from glob import glob
import json
import os

from tqdm import tqdm
import pandas as pd
import numpy as np
import webdataset as wds

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from utils import SGFParseRankPrediction
from network import GoRankResNet

In [2]:
import platform
print(platform.python_version())

3.10.16


In [3]:
torch.__version__

'2.8.0+cu128'

# 0. Quick Look to the Dataset

Change the filepath to your own dataset when you run the code.

In [ ]:
TRAIN_PATH = './dataset/rank_prediction_train.csv'
train_data = pd.read_csv(TRAIN_PATH)
train_data

Here, each row in the dataset corresponds to a player in a match. The columns are as follows:
- `game_id`: Unique identifier for each match.
- `sgf_content`: The SGF content of the match.
- `target_color`: The color (black or white) of the player.
- `rank`: The rank of the player (target variable to predict).

In [ ]:
train_data.info()

In [ ]:
unique_ranks = set(train_data['rank'])
print(f"Total unique ranks: {len(unique_ranks)}")

In [ ]:
# Count of samples per rank
train_data.groupby('rank').size()

There are 8000 samples for each rank from 13k to 7d, totaling 160,000 samples.

In this tutorial, we will predict the `rank` based on the `sgf_content` and `target_color`.

Samples for each rank are divided into training and validation sets as 7000 and 1000 samples, respectively.

In [ ]:
# Create a mapping from rank strings to integer labels
# Labels: 0-19 corresponding to 13k-7d
rank_mapping = {
    "13k": 0, "12k": 1, "11k": 2, "10k": 3, "9k": 4,
    "8k": 5, "7k": 6, "6k": 7, "5k": 8, "4k": 9,
    "3k": 10, "2k": 11, "1k": 12, "1d": 13, "2d": 14,
    "3d": 15, "4d": 16, "5d": 17, "6d": 18, "7d": 19
}
# Save labels to a json file for future reference
JSON_PATH = './rank_labels.json'
with open(JSON_PATH, 'w') as f:
    json.dump(rank_mapping, f, indent=4)

# Load the rank to ID mapping
RANK_TO_ID = json.load(open(JSON_PATH, 'r'))
# Create the inverse mapping from ID to rank
ID_TO_RANK = {v: k for k, v in RANK_TO_ID.items()}

# Initialize feature extractor
sgf_parser = SGFParseRankPrediction()

# Create directories to save extracted features
FEATURES_DIR = f"./rank-features/"
TRAIN_DIR = f"{FEATURES_DIR}/train"
VAL_DIR = f"{FEATURES_DIR}/val"

os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

# 1. Feature Extraction

This section focuses on extracting relevant features from the SGF files to prepare the training dataset.

This section is **one time use** to extract features for training the model.

This process may take some time (approx. an hour) depending on your computer's performance. If you have already extracted the features, you can skip this section and move directly to the model training section.

In [ ]:
NUM_VALIDATION_SAMPLES_PER_RANK = 1000  # Limit the number of validation samples per rank to balance the dataset

In [ ]:
# Select the last NUM_VALIDATION_SAMPLES_PER_RANK rows for each rank for validation
selected_rows = []
for rank in unique_ranks:
    rank_rows = train_data[train_data['rank'] == rank]
    selected_rows.append(rank_rows.tail(NUM_VALIDATION_SAMPLES_PER_RANK))

df_val = pd.concat(selected_rows).reset_index(drop=True)
# remove the items in df_val to create df_train
df_train = pd.concat([train_data, df_val, df_val]).drop_duplicates(keep=False).reset_index(drop=True)
# shuffle df_train
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

df_train should be having 140,000 samples and shuffled. df_val should be having 20,000 samples.

In [ ]:
df_train

In [ ]:
df_val

To prepare features, we use sgfmill library to parse SGF files and extract board states

The details of feature extraction are implemented in `utils.py` file. We use the `SGFParseRankPrediction` class from this file to handle the parsing and feature extraction.

In [ ]:
# Example:
sgf_content = '(;CA[UTF-8]HD[1]RE[W+3.5];B[qd];W[dp];B[pp];W[cc];B[dd];W[dc];B[ec];W[eb];B[fc];W[fb];B[gc];W[cd];B[de];W[cf];B[ce];W[be];B[df];W[cg];B[dg];W[dh];B[eh];W[di];B[ei];W[dj];B[gb];W[od];B[mc];W[ne];B[pf];W[ld];B[lc];W[jd];B[kc];W[kd];B[jc];W[id];B[ic];W[og];B[pg];W[oh];B[ph];W[pi];B[oi];W[qi];B[oj];W[qk];B[of];W[nf];B[ng];W[nh];B[mg];W[mh];B[lg];W[lh];B[kg];W[kh];B[jg];W[qc];B[jh];W[qe];B[ki];W[pd];B[pk];W[ql];B[pm];W[qm];B[pn];W[ro];B[qn];W[rn];B[cn];W[co];B[en];W[fq];B[hq];W[pq];B[oq];W[qq];B[qp];W[rp];B[fp];W[gq];B[gp];W[hr];B[iq];W[ir];B[ep];W[dq];B[do];W[cp];B[jr];W[ej];B[is];W[or];B[eq];W[er];B[gr];W[fr];B[hs];W[nq];B[cr];W[dr];B[bq];W[bn];B[cm];W[bm];B[cl];W[bl];B[ck];W[bp];B[bk];W[bs];B[bh];W[ch];B[bi];W[op];B[rg];W[rd];B[hd];W[he];B[ge];W[if];B[hf];W[ie];B[ig];W[nn];B[nm];W[mn];B[mm];W[ln];B[lm];W[kn];B[km];W[gj];B[jn];W[fi];B[fh];W[gh];B[gg];W[nc];B[md];W[me];B[nd];W[oe];B[oc];W[qg];B[qh];W[rh];B[qf];W[ri];B[rf];W[pc];B[nb];W[pb];B[li];W[jo];B[hh];W[in];B[gi];W[jm];B[fj];W[kq];B[kr];W[lq];B[jl];W[il];B[jk];W[lr];B[ks];W[hp];B[gn];W[fk];B[hj];W[gk];B[fa];W[cb];B[ea];W[db];B[bg];W[bf];B[hk];W[ik];B[ij];W[fm];B[fn];W[em];B[dm];W[cj];B[bj];W[hl];B[ek];W[dk];B[gl];W[el];B[hm];W[im];B[gm];W[da];B[ga];W[ip];B[jq];W[jp];B[ho];W[on];B[om];W[pl];B[ol];W[pj];B[ok];W[ob];B[nc];W[le];B[re];W[pe];B[af];W[ae];B[ag];W[ac];B[fs];W[es];B[gs];W[ls];B[sh];W[si];B[sg];W[se];B[al];W[am];B[ak];W[ao];B[oo];W[no];B[io];W[po];B[qo];W[na];B[ma];W[oa];B[fi];W[sd];B[mi];W[jf];B[kf];W[ke];B[oo];W[mb];B[lb];W[po];B[hn];W[jn];B[oo];W[la];B[ka];W[po];B[rc];W[rb];B[oo];W[gd];B[fd];W[po];B[rj];W[qj];B[oo];W[gf];B[ff];W[po];B[rm];W[rl];B[oo];W[sf];B[ni];W[po];B[sc];W[sb];B[oo];W[eo];B[fo];W[po];B[fl];W[ek];B[oo];W[hc];B[hb];W[po];B[bb];W[oo];B[ci];W[bc];B[dl];W[ba])'

feature, err = sgf_parser.features_from_sgf_content(sgf_content)
if err is not None:
    raise err
feature.shape

Use webdataset library to create a webdataset for efficient data loading during model training and validation

In [ ]:
def extract_and_save_webdataset(df, rank_mapping, output_pattern, samples_per_shard):
    """
    Extract features from dataframe and save as WebDataset format
    Args:
        df: DataFrame with sgf content and ranks
        rank_mapping: Dictionary mapping rank strings to integers
        output_pattern: Pattern for output tar files (e.g., 'train-%06d.tar')
        samples_per_shard: Number of samples per shard file
    """
    total_samples = 0
    
    with wds.ShardWriter(output_pattern, maxcount=samples_per_shard) as sink:
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {output_pattern.split('/')[-1].split('-')[0]}"):
            sgf_content = row['sgf_content']
            rank = row['rank']
            game_name = row['game_id']
            
            try:
                # Extract features from SGF content
                feature, err = sgf_parser.features_from_sgf_content(sgf_content)
                if err is not None:
                    raise err
                sink.write({ # Write feature and label to the webdataset
                    "__key__": f"{game_name}",
                    "features.npy": feature,
                    "label.txt": str(rank),
                    "rank_idx.txt": str(rank_mapping[rank]),
                    "game_name.txt": str(game_name)
                })
                total_samples += 1
            except Exception as e:
                print(f"Error processing game {game_name} with rank {rank}: {e}")
                continue
    
    print(f"Total samples saved: {total_samples}")
    return total_samples

Below is the code to extract features and save them into webdataset format.

It might take some time to run this code (approx 1 hour). You can see the progress in the output logs.

In [ ]:
# Extract and save training set
print("Processing training dataset...")
train_pattern = f'{TRAIN_DIR}/train-%06d.tar'
train_samples = extract_and_save_webdataset(df_train, rank_mapping, train_pattern, 1000)

In [ ]:
# Extract and save validation set
print("Processing validation dataset...")
val_pattern = f'{VAL_DIR}/val-%06d.tar'
val_samples = extract_and_save_webdataset(df_val, rank_mapping, val_pattern, 1000)

# 2. Data Loader

After the features are extracted and saved in webdataset format, we can create data loaders for training and validation.

**If you have already extracted the features, you can skip Section 1 feature extraction and move directly to this section.**

In [ ]:
# CONFIGS
# Data loader config
BATCH_SIZE = 256 # Adjust based on your GPU memory

# Model training config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "./rank-trained-models/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# Training hyperparameters
LEARNING_RATE = 5e-3
NUM_EPOCHS = 50

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Number of epochs: {NUM_EPOCHS}")

In [ ]:
tar_files_train = sorted(glob(f'{TRAIN_DIR}/train-*.tar'))
tar_files_val = sorted(glob(f'{VAL_DIR}/val-*.tar'))

print(f"Found {len(tar_files_train)} training tar files")
print(f"Found {len(tar_files_val)} validation tar files")

In [ ]:
def decode_sample(sample):
    """Decode a sample from WebDataset"""
    features = sample['features.npy']
    rank_idx = int(sample['rank_idx.txt'])
    game_name = sample['game_name.txt']
    
    return (
        torch.from_numpy(features).float(),
        torch.tensor(rank_idx, dtype=torch.long),
        game_name
    )

In [ ]:
train_dataset = (
    wds.WebDataset(tar_files_train, shardshuffle=1)
    .shuffle(1000)
    .decode()  # This properly decodes the data including numpy arrays
    .map(lambda x: decode_sample(x))
    .batched(BATCH_SIZE, partial=True)
)

# Create train DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=None,  # Already batched
)


In [ ]:
val_dataset = (
    wds.WebDataset(tar_files_val, shardshuffle=0) # No need to shuffle validation data
    .decode()  # This properly decodes the data including numpy arrays
    .map(lambda x: decode_sample(x))
    .batched(BATCH_SIZE, partial=True)
)

# Create validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=None,  # Already batched
)

In [ ]:
# Get a batch to verify everything works
batched_data, batch_labels, game_name = next(iter(val_loader)) 
print(batched_data.shape, batch_labels)

# 3. Model Training

Details of the model architecture are implemented in `network.py` file. We use the `GoRankResNet` class from this file to define our model.

`GoRankResNet` is a residual network designed for rank prediction in Go. It consists of an initial convolutional layer, followed by multiple residual blocks, and ends with fully connected layers to output the rank predictions. For regularization, dropout layers are included before the final fully connected layers. As for the reference and baseline in this competition, we keep the architecture relatively simple yet effective. You are welcome to modify the architecture as needed. There is no restriction on the model architecture in this competition.

In [ ]:
model = GoRankResNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Model initialized: {num_params:,} parameters")
print(f"Optimizer: AdamW (lr={LEARNING_RATE})")

In [ ]:
def validate(model, val_loader, criterion, device):
    """
    Run validation on the validation dataset
    Args:
        model: The model to validate
        val_loader: Validation data loader
        criterion: Loss function
    Returns:
        Tuple of (average_loss, accuracy)
    """
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    num_batches = 0
    
    with torch.no_grad():
        for inputs, labels, _ in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            num_batches += 1
    
    avg_loss = val_loss / num_batches if num_batches > 0 else 0
    accuracy = 100 * correct / total if total > 0 else 0
    
    model.train()
    return avg_loss, accuracy

In [ ]:
# Training loop
model.train()
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f'Starting epoch {epoch + 1}/{NUM_EPOCHS}...')
    total_loss = 0.0
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0
    
    pbar = tqdm(enumerate(train_loader), desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    
    for i, (inputs, labels, game_names) in pbar:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_preds += labels.size(0)
        correct_preds += (predicted == labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{running_loss / (i + 1):.4f}',
            'acc': f'{100 * correct_preds / total_preds:.2f}%'
        })
    pbar.close()

    # Training result of the epoch
    train_acc = 100 * correct_preds / total_preds
    avg_epoch_loss = total_loss / (i + 1)
    print(f'Epoch [{epoch + 1}/{NUM_EPOCHS}] Training completed.')
    print(f'Training - Average Loss: {avg_epoch_loss:.4f}, Accuracy: {train_acc:.2f}%')
    
    # Run validation at end of epoch
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    print(f'Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%')
    
    # Save checkpoint if best validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{CHECKPOINT_DIR}best_model.pth')
        print(f'New best validation accuracy: {best_val_acc:.2f}% - Model saved!')
    
    # Save epoch checkpoint
    torch.save(model.state_dict(), f'{CHECKPOINT_DIR}epoch_{epoch+1}.pth')
    print("-----------------------------------------------------")

print("Training completed!")
print(f"Best validation accuracy achieved: {best_val_acc:.2f}%")

# 4. Validation

In [ ]:
# Visualize confusion matrix on validation set
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
model_path = f'{CHECKPOINT_DIR}best_model.pth'
model = GoRankResNet().to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))

In [ ]:
# Validation Loop
total_games = 0
correct_predictions = 0
corect_predictions_pm_one = 0 # Correct predictions for 1+- classes

# Confusion Matrix Data
all_true = []
all_pred = []

with torch.no_grad():
    model.eval()
    for batch_idx, (features, rank_idxs, game_names) in enumerate(val_loader):
        features = features.to(DEVICE)
        rank_idxs = rank_idxs.to(DEVICE)
        
        outputs = model(features).softmax(dim=1)
        _, predicted = torch.max(outputs, 1)

        # Store for confusion matrix
        all_true.extend(rank_idxs.cpu().numpy())
        all_pred.extend(predicted.cpu().numpy())

        # Check predictions within ±1 of the true rank
        correct_plus_one = (predicted == rank_idxs + 1).sum().item()
        correct_minus_one = (predicted == rank_idxs - 1).sum().item()
        correct_preds = (predicted == rank_idxs).sum().item()
        corect_predictions_pm_one += correct_plus_one + correct_minus_one + correct_preds
        
        total_games += rank_idxs.size(0)
        correct_predictions += (predicted == rank_idxs).sum().item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Processed {total_games} games...")
    
    accuracy = correct_predictions / total_games * 100
    accuracy_pm_one = corect_predictions_pm_one / total_games * 100
    print('--- Validation Results ---')
    print(f"Validation Accuracy: {accuracy:.2f}%")
    print(f"Validation Accuracy (±1 rank): {accuracy_pm_one:.2f}%")
    print(f"Total Games Evaluated: {total_games}")
    print(f"Total Correct Predictions (±1 rank): {corect_predictions_pm_one}")
    print(f"Only Same Rank Correct Predictions: {correct_predictions}")
    print(f"Only ±1 Rank Correct Predictions: {corect_predictions_pm_one - correct_predictions}")


In [ ]:
JSON_PATH = './rank_labels.json'
RANK_TO_ID = json.load(open(JSON_PATH, 'r'))
ID_TO_RANK = {v: k for k, v in RANK_TO_ID.items()}

# Confusion Matrix
cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[str(ID_TO_RANK[i]) for i in range(20)],
            yticklabels=[str(ID_TO_RANK[i]) for i in range(20)])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# 5. Inference with Testing Data On-the-Fly

In [ ]:
JSON_PATH = './rank_labels.json'
RANK_TO_ID = json.load(open(JSON_PATH, 'r'))
ID_TO_RANK = {v: k for k, v in RANK_TO_ID.items()}

model_path = f'{CHECKPOINT_DIR}best_model.pth'
model = GoRankResNet().to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))

In [ ]:
sgf_parser = SGFParseRankPrediction()

TEST_PATH = './dataset/rank_prediction_test_sample.csv'
testing_data = pd.read_csv(TEST_PATH)
testing_data

In [ ]:
# Example:
sgf_content = '(;CA[UTF-8]HD[1]RE[B+R];B[pd];W[dd];B[pq];W[dp];B[qj];W[dj];B[fc];W[dc];B[jd];W[dg];B[cm];W[co];B[bj];W[ck];B[bk];W[cl];B[bl];W[dm];B[cn];W[dn];B[di];W[ci];B[cj];W[ei];B[dk];W[dl];B[ej];W[dh];B[dj];W[ch];B[fi];W[eh];B[gj];W[bo];B[fq];W[hq];B[gp];W[jq];B[hp];W[ip];B[gm];W[mq];B[qo];W[eq];B[fr];W[bm];B[eb];W[hc];B[gd];W[hd];B[he];W[id];B[ie];W[jc];B[kd];W[kc];B[ld];W[lc];B[md];W[gb];B[fe];W[gg];B[mc];W[lb];B[mb];W[ib];B[db];W[ig];B[jg];W[jh];B[kg];W[kh];B[lg];W[hi];B[qd];W[qg];B[ph];W[pg];B[qh];W[ng];B[og];W[oh];B[of];W[nf];B[nh];W[mh];B[oi];W[mg];B[lh];W[mi];B[li];W[mj];B[lj];W[nk];B[lk];W[pl];B[rl];W[qm];B[rm];W[qn];B[rn];W[po];B[pp];W[oo];B[np];W[op];B[nq];W[oq];B[or];W[qp];B[mr];W[ro];B[qr];W[qq];B[pr];W[rr];B[rs];W[sr];B[lq];W[ne];B[oe];W[nd];B[oc];W[nc];B[ob];W[nb];B[na];W[la];B[ma];W[im];B[bc];W[bd];B[cc];W[cd];B[gh];W[hh];B[fg];W[fh];B[gf];W[gi];B[fj];W[hj];B[in];W[hm];B[hn];W[gl];B[fl];W[fm];B[gn];W[gk];B[em];W[fn];B[en];W[fo];B[el];W[bn];B[eo];W[jn];B[fp];W[fk];B[ek];W[km];B[jo];W[ko];B[io];W[kp];B[jp];W[kq];B[iq];W[mp];B[no];W[lr];B[mo];W[lp];B[ln];W[ms];B[nr];W[lm];B[kn];W[jm];B[mm];W[ml];B[nm];W[ir];B[hr];W[jr];B[ll];W[lo];B[mk];W[mn];B[nl];W[oj];B[ok];W[nn];B[on];W[ln];B[nj];W[om];B[ad];W[ae];B[ac];W[bf];B[cs];W[cr];B[ds];W[bs];B[dr];W[cq];B[br];W[er];B[es];W[bq];B[as];W[dq];B[fs];W[ip];B[gq];W[iq];B[go];W[hs];B[gs];W[gr];B[hk];W[hl];B[hr];W[if];B[jf];W[qk];B[rk];W[pk];B[pj];W[ik];B[is];W[js];B[hs];W[bi];B[ef];W[df];B[ee];W[ed];B[fd];W[fb];B[ec];W[eg];B[de];W[ce];B[hf];W[hg];B[ff];W[jj];B[ol];W[pm];B[kj];W[ki];B[jk];W[jl];B[kk];W[fa];B[ea];W[ai];B[aj];W[al];B[ak];W[am];B[so];W[sp];B[sn];W[aq];B[bs];W[do];B[gc])'
rank_id = None

feature, err = sgf_parser.features_from_sgf_content(sgf_content)
if err is not None:
    raise err
feature = torch.from_numpy(feature).float().to(DEVICE)

with torch.no_grad():
    model.eval()
    outputs = model(feature.unsqueeze(0)).softmax(dim=1)
    _, predicted = torch.max(outputs, 1)
    rank_id = predicted.item()

print(f"Predicted rank index: {rank_id}, Rank: {ID_TO_RANK[rank_id]}")

In [ ]:
for idx, row in testing_data.iterrows():
    sgf_content = row['sgf_content']
    rank_id = None
    
    feature, err = sgf_parser.features_from_sgf_content(sgf_content)
    if err is not None:
        raise err
    feature = torch.from_numpy(feature).float().to(DEVICE)

    with torch.no_grad():
        model.eval()
        outputs = model(feature.unsqueeze(0)).softmax(dim=1)
        _, predicted = torch.max(outputs, 1)
        rank_id = predicted.item()
    # Update the dataframe with predicted rank
    testing_data.at[idx, 'rank'] = ID_TO_RANK[rank_id]

In [ ]:
testing_data

In [ ]:
# Save the csv file as submission-rank.csv
testing_data.to_csv('submission-rank.csv', index=False)

# End of the tutorial

This concludes the rank prediction tutorial. The trained best model of this tutorial will be shared. You should be able to get around 17% and 37% accuracy rates for same rank and +-1 in the testing set, respectively.

The tutorial is prepared by Serkan Kavak, NDHU AI Lab. If you have any questions or need further assistance, feel free to open an issue on the project's GitHub repository. Good luck with your rank prediction model!